In [2]:
import networkx as nx 
from dataXplorer import JupyterClient
from dataXplorer import AnalysisToolkit, SessionManager, TextureGenerator, ProjectFileManager, VisualizerSyncer

import pandas as pd

### script specific functions

In [3]:
import numpy as np  
import random 

def generate_peripheral_position():
    coords = []
    for _ in range(3):
        edge_val = np.random.choice([0.0, 1.0])
        jitter = np.random.uniform(-0.1, 0.1)
        coords.append(np.clip(edge_val + jitter, 0.0, 1.0))
    return tuple(coords)
    
def generate_random_central_position():
    coords = []
    for _ in range(3):
        edge_val = np.random.choice([0.49, 0.51])
        jitter = np.random.uniform(-0.001, 0.001)
        coords.append(np.clip(edge_val + jitter, 0.49,0.51))
    return tuple(coords)

def generate_random_spherical_position(spatial_range=(0.8, 1.0)):
        radius = random.uniform(*spatial_range)
        theta = random.uniform(0, 2 * np.pi)
        phi = random.uniform(0, np.pi)
        x = radius * np.sin(phi) * np.cos(theta)
        y = radius * np.sin(phi) * np.sin(theta)
        z = radius * np.cos(phi)
        
        # Normalize to 0-1 range
        x = (x + 1) / 2
        y = (y + 1) / 2
        z = (z + 1) / 2
        
        return (x, y, z)

### Connect

In [4]:
# run backend (server) using buildandrun powershell script
client = JupyterClient()

ConnectionError: Connection refused by the server

In [ ]:
#client.disconnect()

# Prerequisites & Infos before starting: 

In [ ]:
# FIRST : upload a Graph with all needed information as node annotation attributes
# (e.g. this can include initial positions in case you want to access them and 
# for time reasons start off with a precalculated layout, node types if there are 
# several graphs merged into one, ... )

# then use the function "make_json(G)" in nx2json and upload the graph.
# this step could be integrated into the notebook though. 
 
# then you can run the cells in this notebook one by one. 

# Note: input files are stored locally and not shared via git (temp-files folder)

# Get started

In [ ]:
# see if new project is in projectlist
import GlobalData as GD

allprojects_updated = []
for i, proj in enumerate(GD.listProjects()):
    allprojects_updated.append((i,proj))
allprojects_updated

[(0, 'CDK5'),
 (1, 'CircLadderGraph-xsmall'),
 (2, 'CLGraph_TEST'),
 (3, 'diffusion'),
 (4, 'Exposurome'),
 (5, 'GenExpression_01'),
 (6, 'GenExpression_02'),
 (7, 'imunet_250130-X0-inter'),
 (8, 'imunet_250130-X1-inter'),
 (9, 'Interactive_Project_T01'),
 (10, 'JSON_autocore'),
 (11, 'JSON_barbellgraph'),
 (12, 'JSON_Zachary'),
 (13, 'Microplastics_HumanHealth'),
 (14, 'Powergrid_Europe'),
 (15, 'Realtime-project'),
 (16, 'Sphere_Torus'),
 (17, 'Sphere_Torus_Morph'),
 (18, 'Teapot'),
 (19, 'TEAPOT-RealtimeTesting'),
 (20, 'TheMandelbulb_edges')]

In [ ]:
# select a project to work with
sel_id = 13
sel_name = allprojects_updated[sel_id][1]

# load the project data
session = SessionManager(sel_id, sel_name, client)
session.load_graph_from_project()

session.reload_project()

# initialize 
file_mgr = ProjectFileManager(session)
tex_gen = TextureGenerator(session)
syncer = VisualizerSyncer(session)
tools = AnalysisToolkit(session, tex_gen, syncer, file_mgr)

Session Graph loaded from project folder. Data: Nodes: 16193 Links: 14191


In [ ]:
latest_message = client.latest_data
session.get_active_layouts(latest_message)

AttributeError: 'NoneType' object has no attribute 'get'

### Reconstruct Graph from Project

In [ ]:
G = session.load_graph_from_project()
print("G_nodes:", len(G.nodes()))
print("G_edges:", len(G.edges()))

Session Graph loaded from project folder. Data: Nodes: 16193 Links: 14191
G_nodes: 16193
G_edges: 14191


In [5]:
# check for node attributes 
node_attr = G.nodes(data=True)

# quick check
node_attr[1]

NameError: name 'G' is not defined

In [6]:
# get attr "init-pos" and set to Graph nodes attributes "pos"

pos = {}
for node in G.nodes():
    pos[node] = node_attr[node]["attrlist"]["init_pos"]

# quick check
pos[0]

NameError: name 'G' is not defined

In [7]:
nx.set_node_attributes(G, pos, 'pos')

NameError: name 'G' is not defined

In [8]:
node_type = {}
for node in G.nodes():
    node_type[node] = node_attr[node]["attrlist"]["type"]

NameError: name 'G' is not defined

### lat lon to x y z 

In [9]:
df = pd.read_csv("temp-files/Microplastics/whc-sites(tangibles)-2021.csv")
df

,Name,short_description,date_inscribed,danger,date_end,longitude,latitude,area_hectares,category_long,category_short,Country name,Region,iso_code,transboundary,rev_bis
0,L’Anse aux Meadows National Historic Site,<p>At the tip of the Great Northern Peninsula ...,1978,0,NaN,-55.616667,51.466667,7991.00,Cultural,C,Canada,Europe and North America,ca,0,NaN
1,Nahanni National Park,"<p>Located along the South Nahanni River, one ...",1978,0,NaN,-125.589444,61.547222,476560.00,Natural,N,Canada,Europe and North America,ca,0,NaN
2,Galápagos Islands,"<p>Situated in the Pacific Ocean some 1,000 km...",1978,0,2010.0,-90.501319,-0.689860,14066514.00,Natural,N,Ecuador,Latin America and the Caribbean,ec,0,Bis
3,City of Quito,"<p>Quito, the capital of Ecuador, was founded ...",1978,0,NaN,-78.512083,-0.220000,70.43,Cultural,C,Ecuador,Latin America and the Caribbean,ec,0,NaN
4,Simien National Park,<p>Massive erosion over the years on the Ethio...,1978,0,2017.0,38.066667,13.183333,13600.00,Natural,N,Ethiopia,Africa,et,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,The work of engineer Eladio Dieste: Church of ...,<p>The Church of Atlántida with its belfry and...,2021,0,NaN,-55.766408,-34.743919,0.56,Cultural,C,Uruguay,Latin America and the Caribbean,uy,0,NaN
1151,The Great Spa Towns of Europe,The transnational site of The Great Spa Towns ...,2021,0,NaN,5.866944,50.492222,7014.00,Cultural,C,"Austria,Belgium,Czechia,France,Germany,Italy,U...",Europe and North America,"at,be,cz,de,fr,gb,it",1,NaN
1152,Frontiers of the Roman Empire – The Danube Lim...,<p>It covers almost 600km of the whole Roman E...,2021,0,NaN,16.861389,48.115194,NaN,Cultural,C,"Austria,Germany,Slovakia",Europe and North America,"at,de,sk",1,NaN
1153,Colonies of Benevolence,<p>The transnational serial property encompass...,2021,0,NaN,6.391589,53.042222,2012.00,Cultural,C,"Belgium,Netherlands",Europe and North America,"be,nl",1,NaN


In [13]:
import pandas as pd

from math import radians, cos, sin, sqrt, atan, atan2
def geodetic_to_geocentric(ellipsoid, lat, lon, height):
    # Adjust the flattening factor to influence the ellipsoid shape
    a, rf = ellipsoid
    # rf *= 1.2  # Increase the flattening factor to bring poles closer together

    phi = radians(lat)
    lamb = radians(lon)

    sin_phi = sin(phi)
    e2 = 1 - (1 - 1 / rf) ** 2
    n = a / sqrt(1 - e2 * sin_phi ** 2) 
    
    r = ((n + height) * cos(phi))
    x = r * cos(lamb)
    y = r * sin(lamb)   
    z = (n * (1 - e2) + height) * sin(phi) 

    return x, y, z  


def normalize_coordinates(x, y, z):
    magnitude = sqrt(x**2 + y**2 + z**2)
    x_norm, y_norm, z_norm = x / magnitude, y / magnitude, z / magnitude
    x_final = (x_norm+1)/2
    y_final = (y_norm+1)/2
    z_final = (z_norm+1)/2
    return x_final, y_final, z_final


df = pd.read_csv("temp-files/Microplastics/whc-sites(tangibles)-2021v")
pos_latlon = df[["Latitude", "Longitude"]].values.tolist()
d_pos_latlon = dict(zip(df["Country"], pos_latlon))


# modify to rotate 90 degrees into the other direction around z-axis
d_pos_latlon_rotated = {}
for k, v in d_pos_latlon.items():
    lat, lon = v
    # Rotate the longitude by -90 degrees
    lon_rotated = (lon - 90) % 360 #- 90
    d_pos_latlon_rotated[k] = (lat, lon_rotated)

pos_xyz = {}

flattening_factor = 298.257223563
WGS84_radius = 6378137.0

for k,v in d_pos_latlon_rotated.items():
    x,y,z = geodetic_to_geocentric((WGS84_radius, flattening_factor), v[0],v[1], 0)
    xn, yn, zn = normalize_coordinates(x,y,z)
    pos_xyz[k] = (yn,xn,zn)

# fix NaN values in init_pos
for i, k in pos_xyz.items():
    if np.isnan(k[0]) or np.isnan(k[1]) or np.isnan(k[2]):
        pos_xyz[i] = generate_random_spherical_position(spatial_range=(0.8, 1.0))

FileNotFoundError: [Errno 2] No such file or directory: 'temp-files/Microplastics/whc-sites(tangibles)-2021v'

# SCENES

### SCENE 1 - unesco world heritage sites + globe 

In [118]:
unesco_nodes = []
for node, attr in node_attr:
    if 'unesco' in node_type[node]:
        unesco_nodes.append(node)

print("unesco_nodes:", len(unesco_nodes))

unesco_nodes: 1155


In [119]:
# node positions 
    
pos_unesco = {}
for i in G.nodes():
    if i in unesco_nodes:
        pos_unesco[i] = G.nodes[i]['pos']
    else:
        pos_unesco[i] = generate_random_spherical_position(spatial_range=(0.0, 0.5))

In [120]:
# adapt positions for globe 
pos_unesco_2 = {}
for k,v in pos_unesco.items():
    x,y,z = v 
    pos_unesco_2[k] = (-y,x,z)

pos_unesco = pos_unesco_2

✅ Connected to /main


In [121]:
# node colors 

col_cultural = (185,103,0,120)
col_natural = (91,231,0,200)
col_others = (131,144,70,90)

nodecol_unesco = {}
for node in G.nodes():
    if "Cultural" in node_attr[node]["attrlist"]["type"]:
        nodecol_unesco[node] = col_cultural
    elif "Natural" in node_attr[node]["attrlist"]["type"]:
        nodecol_unesco[node] = col_natural
    else:
        nodecol_unesco[node] = (0,0,0,0)

In [122]:
linkcol_unesco = {} # empty : all edges should be black 

In [123]:
layout_name = "scene01-UNESCO-sites_geo"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_unesco, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_unesco, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_unesco, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 2 - Blood system

In [124]:
# node colors 

col_blood_rgba = (250,84,0,100) 

blood_nodes = []
for node, attr in node_attr:
    if 'BloodSystem-' in attr['name']:
        blood_nodes.append(node)

nodecol_bloodsystem = {}
for i in G.nodes():
    if i in blood_nodes:
       nodecol_bloodsystem[i] = col_blood_rgba
    else:
        nodecol_bloodsystem[i] = (0,0,0,0)

In [125]:
# node positions 

ppi_nodes = []
for node, attr in node_attr:
    if 'ppi' in node_type[node]:
        ppi_nodes.append(node)
        

pos_organs = {}
for i in G.nodes():
    if i in blood_nodes:
        pos_organs[i] = G.nodes[i]['pos']
    elif i in ppi_nodes:
        pos_organs[i] = G.nodes[i]['pos'] # generate_random_spherical_position()
    else:
        pos_organs[i] = generate_random_central_position()

In [126]:
# show min and max values of x y z of posG_organs
pos_organs_array = np.array(list(pos_organs.values()))
print("min x:", np.min(pos_organs_array[:,0]))
print("max x:", np.max(pos_organs_array[:,0]))
print("min y:", np.min(pos_organs_array[:,1]))
print("max y:", np.max(pos_organs_array[:,1]))
print("min z:", np.min(pos_organs_array[:,2]))
print("max z:", np.max(pos_organs_array[:,2]))

min x: 0.0
max x: 1.0
min y: 0.0
max y: 1.0
min z: 0.0010639552219921522
max z: 0.9999999999999998


In [127]:
# link colors 

linkcol_bloodsystem = {}

l_blood_edges = []

for i in G.edges():
    if i[0] in blood_nodes and i[1] in blood_nodes:
       l_blood_edges.append(i)
       linkcol_bloodsystem[i] = col_blood_rgba

    else:
        linkcol_bloodsystem[i] = (0,0,0,0)

In [128]:
layout_name = "scene02-Organs-Bloodsystem"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_bloodsystem, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_bloodsystem, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_organs, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 3 - Blood System + Liver+Kidneys

In [129]:
# node colors 

col_liver_rgba = (112,45,0,90)
col_kidney_rgba = (142,0,0,100)

liver_nodes = []
for node, attr in node_attr:
    if 'Liver-' in attr['name']:
        liver_nodes.append(node)
kidney_nodes = []
for node, attr in node_attr:
    if 'Kidneys-' in attr['name']:
        kidney_nodes.append(node)

nodecol_blood_liver_kidneys = {}
for i in G.nodes():
    if i in liver_nodes:
        nodecol_blood_liver_kidneys[i] = col_liver_rgba
    elif i in kidney_nodes:
        nodecol_blood_liver_kidneys[i] = col_kidney_rgba
    elif i in blood_nodes:
       nodecol_blood_liver_kidneys[i] = col_blood_rgba
    else:
        nodecol_blood_liver_kidneys[i] = (0,0,0,0)

In [130]:
pos_organs = {}
for i in G.nodes():
    if i in liver_nodes or i in kidney_nodes or i in blood_nodes:
        pos_organs[i] = G.nodes[i]['pos']
    elif i in ppi_nodes:
        pos_organs[i] =  G.nodes[i]['pos']  #generate_random_spherical_position()
    else:
        pos_organs[i] = generate_random_central_position()

In [131]:
# link colors 

linkcol_blood_liver_kidneys = {}

l_liver_edges = []
l_kidney_edges = []
l_blood_edges = []

for i in G.edges():
    if i[0] in liver_nodes and i[1] in liver_nodes:
        l_liver_edges.append(i)
        linkcol_blood_liver_kidneys[i] = col_liver_rgba

    elif i[0] in kidney_nodes and i[1] in kidney_nodes:
        l_kidney_edges.append(i)
        linkcol_blood_liver_kidneys[i] = col_kidney_rgba

    elif i[0] in blood_nodes and i[1] in blood_nodes:
       l_blood_edges.append(i)
       linkcol_blood_liver_kidneys[i] = col_blood_rgba

    else:
        linkcol_blood_liver_kidneys[i] = (0,0,0,0)



In [132]:
layout_name = "scene03-Organs-Bloodsystem-Liver-Kidneys"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_blood_liver_kidneys, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_blood_liver_kidneys, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_organs, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


In [133]:
session.reload_project()

### SCENE 4 - all organs (previous + heart)

In [134]:
# node colors 

col_heart_rgba = (228,9,9,100)

heart_nodes = []
for node, attr in node_attr:
    if 'Heart-' in attr['name']:
        heart_nodes.append(node)

nodecol_organs = {}
for i in G.nodes():
    if i in liver_nodes:
        nodecol_organs[i] = col_liver_rgba
    elif i in heart_nodes:
        nodecol_organs[i] = col_heart_rgba
    elif i in kidney_nodes:
        nodecol_organs[i] = col_kidney_rgba
    elif i in blood_nodes:
       nodecol_organs[i] = col_blood_rgba
    else:
        nodecol_organs[i] = (0,0,0,0)

In [135]:
pos_organs = {}
for i in G.nodes():
    if i in liver_nodes or i in heart_nodes or i in kidney_nodes or i in blood_nodes:
        pos_organs[i] = G.nodes[i]['pos']
    elif i in ppi_nodes:
        pos_organs[i] =  G.nodes[i]['pos'] #generate_random_spherical_position()
    else:
        pos_organs[i] = generate_random_central_position()

In [136]:
# link colors 

linkcol_organs = {}

l_liver_edges = []
l_heart_edges = []
l_kidney_edges = []
l_blood_edges = []

for i in G.edges():
    if i[0] in liver_nodes and i[1] in liver_nodes:
        l_liver_edges.append(i)
        linkcol_organs[i] = col_liver_rgba
    
    elif i[0] in heart_nodes and i[1] in heart_nodes:
        l_heart_edges.append(i)
        linkcol_organs[i] = col_heart_rgba

    elif i[0] in kidney_nodes and i[1] in kidney_nodes:
        l_kidney_edges.append(i)
        linkcol_organs[i] = col_kidney_rgba

    elif i[0] in blood_nodes and i[1] in blood_nodes:
       l_blood_edges.append(i)
       linkcol_organs[i] = col_blood_rgba

    else:
        linkcol_organs[i] = (0,0,0,0)



In [137]:
layout_name = "scene04-Organs-all"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_organs, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_organs, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_organs, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


BadNamespaceError: /main is not a connected namespace.

### SCENE 5 - BPA in center + PPI surrounding

In [ ]:
bpa_nodes = []
for node, attr in node_attr:
    if 'bpa' in node_type[node]:
        bpa_nodes.append(node)
print(len(bpa_nodes))

500


In [ ]:
# node positions

pos_bpa = {}
for i in G.nodes():
    if i in bpa_nodes or i in ppi_nodes:
        pos_bpa[i] = G.nodes[i]['pos']
    else:
        pos_bpa[i] = generate_random_spherical_position(spatial_range=(0.0,0.001))

In [ ]:
# node colors

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib as mpl    

col_palette = cm.get_cmap('Blues', len(ppi_nodes))
norm = mpl.colors.Normalize(vmin=0, vmax=len(ppi_nodes)-1)
cmap = mpl.cm.ScalarMappable(norm=norm, cmap=col_palette)
cmap.set_array([])

d_nodecolors_cmap = {}
for i, node in enumerate(ppi_nodes):
    rgba = cmap.to_rgba(i)
    d_nodecolors_cmap[node] = (rgba[0]*255, rgba[1]*255, rgba[2]*255, 100)

col_bpa_rgba = (220,220,0,100)
    
nodecol_bpa = {}
for i in G.nodes():
    if i in bpa_nodes:
        nodecol_bpa[i] = col_bpa_rgba
    elif i in ppi_nodes:
        nodecol_bpa[i] = d_nodecolors_cmap[i]
    else:
        nodecol_bpa[i] = (0,0,0,0)

C:\Users\chris\AppData\Local\Temp\ipykernel_12944\1639948662.py:7: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  col_palette = cm.get_cmap('Blues', len(ppi_nodes))


In [ ]:
# link colors

linkcol_bpa = {}
l_bpa_edges = []

for i in G.edges():
    if i[0] in bpa_nodes and i[1] in bpa_nodes:
        l_bpa_edges.append(i)
        linkcol_bpa[i] = col_bpa_rgba
    elif i[0] in ppi_nodes and i[1] in ppi_nodes:
        linkcol_bpa[i] = (0,164,215,50)
    else:
        linkcol_bpa[i] = (0,0,0,0)

In [ ]:
layout_name = "scene05-BPA-molecule"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_bpa, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_bpa, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_bpa, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 6 - bpa affected genes highlighted 

In [ ]:
#highlight genes in ppi which are affected by BPA 

bpa_affected_genes = []
with open ("temp-files/Microplastics/BPA_genes.txt", "r") as f:
    for line in f:
        bpa_affected_genes.append(line.strip())
    
print("bpa_affected_genes:", len(bpa_affected_genes))

bpa_affected_genes: 10401


In [ ]:
d_id_nodename = {}
for node, attr in node_attr:
    d_id_nodename[node] = attr['name']

In [ ]:
# node colors 
nodecol_bpa_affected = {}
for i, n in d_id_nodename.items():
    if n in bpa_affected_genes:
        nodecol_bpa_affected[i] = (255,0,0,180)
    elif i in bpa_nodes:
        nodecol_bpa_affected[i] = nodecol_bpa[i]
    elif i in ppi_nodes:
        nodecol_bpa_affected[i] = d_nodecolors_cmap[i]
    else:
        nodecol_bpa_affected[i] = (0,0,0,0)

In [ ]:
layout_name = "scene06-BPA-molecule-affected-PPI-nodes"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_bpa_affected, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_bpa, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_bpa, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 7 - disease modules (cardio..)

In [ ]:
import pickle
with open ('temp-files/Microplastics/BPA_diseases_sigfdr_dict.pickle', 'rb') as f:
    d_disease_genes = pickle.load(f)

for i,n in d_disease_genes.items():
    print(i, len(n))

Alzheimer's Disease 44
Liver carcinoma 43
Seizures 37
Colorectal Carcinoma 77
Familial thoracic aortic aneurysm and aortic dissection 24
Cardiomyopathies 38
Malignant neoplasm of breast 88
Bipolar Disorder 62
Obesity 71
Diabetes Mellitus, Non-Insulin-Dependent 82
Hypertensive disease 100
Liver Cirrhosis, Experimental 44
melanoma 46
Diabetes Mellitus, Experimental 62
Heart failure 21
Congestive heart failure 24
Autistic Disorder 32
Intellectual Disability 125
Myocardial Infarction 33
Schizophrenia 146
Breast Carcinoma 73
Malignant tumor of colon 15
Colorectal Neoplasms 20
Mammary Neoplasms 24
Leukemia, Myelocytic, Acute 53


In [140]:
dismod_1 = list(d_disease_genes["Heart failure"])+list(d_disease_genes["Cardiomyopathies"])
print("len dismod_1:", len(dismod_1))   

dismod_2 = list(d_disease_genes["Hypertensive disease"])
print("len dismod_2:", len(dismod_2))

dismod_3 = list(d_disease_genes["Intellectual Disability"])
print("len dismod_3:", len(dismod_3))

dismod_4 = list(d_disease_genes["Alzheimer's Disease"])#+list(d_disease_genes["Seizures"])
print("len dismod_4:", len(dismod_4))

len dismod_1: 59
len dismod_2: 100
len dismod_3: 125
len dismod_4: 44


In [141]:
d_id_dismod_1 = {}
d_id_dismod_2 = {}
d_id_dismod_3 = {}
d_id_dismod_4 = {}
for k,v in d_id_nodename.items():
    if v in dismod_1:
        d_id_dismod_1[k] = v
    elif v in dismod_2:
        d_id_dismod_2[k] = v
    elif v in dismod_3:
        d_id_dismod_3[k] = v
    elif v in dismod_4:
        d_id_dismod_4[k] = v        

In [142]:
# make subgraphs 

G_dismod_1 = G.subgraph(d_id_dismod_1.keys())
print("G_dismod_1_nodes:", len(G_dismod_1.nodes()))
print("G_dismod_1_edges:", len(G_dismod_1.edges()))

G_dismod_2 = G.subgraph(d_id_dismod_2.keys())
print("G_dismod_2_nodes:", len(G_dismod_2.nodes()))
print("G_dismod_2_edges:", len(G_dismod_2.edges()))

G_dismod_3 = G.subgraph(d_id_dismod_3.keys())
print("G_dismod_3_nodes:", len(G_dismod_3.nodes()))
print("G_dismod_3_edges:", len(G_dismod_3.edges()))

G_dismod_4 = G.subgraph(d_id_dismod_4.keys())
print("G_dismod_4_nodes:", len(G_dismod_4.nodes()))
print("G_dismod_4_edges:", len(G_dismod_4.edges()))

G_dismod_1_nodes: 56
G_dismod_1_edges: 64
G_dismod_2_nodes: 86
G_dismod_2_edges: 125
G_dismod_3_nodes: 81
G_dismod_3_edges: 64
G_dismod_4_nodes: 35
G_dismod_4_edges: 23


In [143]:
# node positions 

pos_diseases_heart = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
        pos_dis_1 = nx.spring_layout(G_dismod_1, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.01)
        pos_dis_1_rescaled = {node: (x*0.1+0.6, y*0.1+0.6, z*0.1+0.6) for node, (x,y,z) in pos_dis_1.items()}
        pos_diseases_heart[i] = pos_dis_1_rescaled[i]
    elif i in d_id_dismod_2.keys():
        pos_dis_2 = nx.spring_layout(G_dismod_2, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.01)
        pos_dis_2_rescaled = {node: (x*0.1+0.4, y*0.1+0.4, z*0.1+0.4) for node, (x,y,z) in pos_dis_2.items()}
        pos_diseases_heart[i] = pos_dis_2_rescaled[i]
    else:
        pos_diseases_heart[i] = pos_bpa[i]  #G.nodes[i]['pos']

In [144]:
# node colors 

col_dismod_1_rgba = (195,3,3,200)   
col_dismod_2_rgba = (255,116,116,200)
col_dismod_3_rgba = (255,100,0,200)
col_dismod_4_rgba = (255,195,120,200) #(121,59,115,200)

nodecol_diseases_heart = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
         nodecol_diseases_heart[i] = col_dismod_1_rgba
    elif i in d_id_dismod_2.keys():
        nodecol_diseases_heart[i] = col_dismod_2_rgba
    elif i in bpa_nodes:
        nodecol_diseases_heart[i] = nodecol_bpa[i]
    elif i in ppi_nodes:
        nodecol_diseases_heart[i] = d_nodecolors_cmap[i]
    else:
        nodecol_diseases_heart[i] = (0,0,0,0)

In [145]:
linkcol_diseases_heart = {}
for i in G.edges():
    if i in l_bpa_edges:
        linkcol_diseases_heart[i] = col_bpa_rgba
    elif i in G_dismod_1.edges():
        linkcol_diseases_heart[i] = col_dismod_1_rgba
    elif i in G_dismod_2.edges():
        linkcol_diseases_heart[i] = col_dismod_2_rgba
    else:
        linkcol_diseases_heart[i] = (0,0,0,0)

In [146]:
layout_name = "scene07-BPA-diseases"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_diseases_heart, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_diseases_heart, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_diseases_heart, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 8 - diseasese (neurological)

In [151]:
# node positions 

pos_diseases_neuro = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
        pos_diseases_neuro[i] = pos_diseases_heart[i]
    elif i in d_id_dismod_2.keys():
        pos_diseases_neuro[i] = pos_diseases_heart[i]
    elif i in d_id_dismod_3.keys():
        pos_dis_3 = nx.spring_layout(G_dismod_3, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.001)
        pos_dis_3_rescaled = {node: (x*0.1+0.2, y*0.1+0.2, z*0.1+0.6) for node, (x,y,z) in pos_dis_3.items()}
        pos_diseases_neuro[i] = pos_dis_3_rescaled[i]
    elif i in d_id_dismod_4.keys():
        pos_dis_4 = nx.spring_layout(G_dismod_4, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.01)
        pos_dis_4_rescaled = {node: (x*0.03+0.6, y*0.03+0.6, z*0.03+0.3) for node, (x,y,z) in pos_dis_4.items()}
        pos_diseases_neuro[i] = pos_dis_4_rescaled[i]

    else:
        pos_diseases_neuro[i] = pos_bpa[i] #G.nodes[i]['pos']

In [152]:

nodecol_diseases_neuro = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
         nodecol_diseases_neuro[i] = col_dismod_1_rgba
    elif i in d_id_dismod_2.keys():
        nodecol_diseases_neuro[i] = col_dismod_2_rgba
    elif i in d_id_dismod_3.keys():
        nodecol_diseases_neuro[i] = col_dismod_3_rgba
    elif i in d_id_dismod_4.keys():
        nodecol_diseases_neuro[i] = col_dismod_4_rgba
    elif i in bpa_nodes:
        nodecol_diseases_neuro[i] = nodecol_bpa[i]
    elif i in ppi_nodes:
        nodecol_diseases_neuro[i] = d_nodecolors_cmap[i]
    else:
        nodecol_diseases_neuro[i] = (0,0,0,0)

In [153]:
linkcol_diseases_neuro = {}
for i in G.edges():
    if i in G_dismod_1.edges():
        linkcol_diseases_neuro[i] = col_dismod_1_rgba
    elif i in G_dismod_2.edges():
        linkcol_diseases_neuro[i] = col_dismod_2_rgba
    elif i in G_dismod_3.edges():
        linkcol_diseases_neuro[i] = col_dismod_3_rgba
    elif i in G_dismod_4.edges():
        linkcol_diseases_neuro[i] = col_dismod_4_rgba
    # elif i[0] in ppi_nodes and i[1] in ppi_nodes:
    #     linkcol_diseases[i] = (0,164,215,50)
    else:
        linkcol_diseases_neuro[i] = (0,0,0,0)

In [ ]:
layout_name = "scene08-BPA-diseases_all"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_diseases_neuro, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_diseases_neuro, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_diseases_neuro, layout_name, save=True)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_pat h_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


✅ Connected to /main


In [139]:
session.reload_project()

✅ Connected to /main
✅ Connected to /main
✅ Connected to /main✅ Connected to /main

